In [27]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('UCI_Credit_Card.csv')
df.head(2)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1


In [ ]:
df.drop('ID',axis=1,inplace=True) #dropping insignificant column

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   30000 non-null  float64
 1   SEX                         30000 non-null  int64  
 2   EDUCATION                   30000 non-null  int64  
 3   MARRIAGE                    30000 non-null  int64  
 4   AGE                         30000 non-null  int64  
 5   PAY_0                       30000 non-null  int64  
 6   PAY_2                       30000 non-null  int64  
 7   PAY_3                       30000 non-null  int64  
 8   PAY_4                       30000 non-null  int64  
 9   PAY_5                       30000 non-null  int64  
 10  PAY_6                       30000 non-null  int64  
 11  BILL_AMT1                   30000 non-null  float64
 12  BILL_AMT2                   30000 non-null  float64
 13  BILL_AMT3                   300

In [5]:
for col in ['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']:
    df[col]=df[col].astype('category')

In [ ]:
x=df.drop('default.payment.next.month',axis=1) #Independent Features
y=df['default.payment.next.month']  #Dependent Feature

In [7]:
cat_features = x.select_dtypes(include=['object', 'category']).columns.tolist()
num_features1 = x.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
         ("OneHotEncoder", oh_transformer, cat_features),  #Applying one-hot encoding to categorical columns
          ("StandardScaler", numeric_transformer, num_features1) #Applying StandardScaler to numeric columns
    ]
)

In [ ]:
x_scaled=preprocessor.fit_transform(x) #Scaling the data

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score

In [ ]:
#Preparing the training and testing dataset 
x_train,x_test,y_train,y_test=train_test_split(x_scaled,y,test_size=0.2,random_state=42)
x_train.shape,y_test.shape

((24000, 82), (6000,))

RANDOM FOREST WITH SCALED DATA

In [12]:
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred=rf.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8200
F1 score:0.7996
Precision:0.6594
Recall:0.3671
ROC:0.6570


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
model = RandomForestClassifier()

# Define hyperparameter grid
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}


# Perform grid search with cross-validation
grid_search = RandomizedSearchCV(estimator=model, param_distributions=rf_params, cv=3,random_state=42, n_jobs=-1)
grid_search.fit(x, y)

# Best hyperparameters
best_params = grid_search.best_params_
print(f"Best parameters found: {best_params}")


Best parameters found: {'n_estimators': 1000, 'min_samples_split': 20, 'max_features': 8, 'max_depth': 5}


In [19]:
rf=RandomForestClassifier(n_estimators=1000,min_samples_split=20,max_features=8,max_depth=5)
rf.fit(x_train,y_train)
y_pred=rf.predict(x_test)
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.8118
F1 score:0.7724
Precision:0.7130
Recall:0.2346
ROC:0.6041


In [ ]:
#Recall is small as the dataset is severely imbalanced

RANDOM FOREST WITH SCALED DATA AND SMOTE

In [20]:
#We don't use SMOTE-NC here as we have already encoded the categorical features using OHE
from imblearn.over_sampling import SMOTE 
sm=SMOTE(random_state=42)

In [ ]:
x_new,y_new=sm.fit_resample(x_train,y_train) #Applying SMOTE

# Define and fit the Random Forest Classifier
model1=RandomForestClassifier()
model1.fit(x_new,y_new)

y_pred_smote=model1.predict(x_test)  # Make predictions on the test set
model_test_accuracy = accuracy_score(y_test, y_pred_smote) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred_smote, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred_smote) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred_smote) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred_smote) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.7967
F1 score:0.7900
Precision:0.5422
Recall:0.4547
ROC:0.6736


In [22]:
from sklearn.model_selection import RandomizedSearchCV
model = RandomForestClassifier()

# Define hyperparameter grid
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}


# Perform grid search with cross-validation
grid_search = RandomizedSearchCV(estimator=model, param_distributions=rf_params, cv=3,random_state=42, n_jobs=-1)
grid_search.fit(x_new, y_new)

# Best hyperparameters
best_params = grid_search.best_params_
print(f"Best parameters found: {best_params}")

Best parameters found: {'n_estimators': 100, 'min_samples_split': 8, 'max_features': 8, 'max_depth': None}


In [23]:
# Define and fit the Random Forest Classifier
rf1=RandomForestClassifier(n_estimators=100,min_samples_split=8,max_features=8,max_depth=None)
rf1.fit(x_new,y_new)

y_pred=rf1.predict(x_test) # Make predictions on the test set
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.7965
F1 score:0.7916
Precision:0.5399
Recall:0.4745
ROC:0.6806


RANDOM FOREST WITH UNDER SAMPLING

In [24]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(x_train, y_train)

# Define and fit the Random Forest Classifier
model = RandomForestClassifier()
model.fit(X_rus, y_rus)

# Make predictions on the test set
y_pred = model.predict(x_test)

# Calculate metrics on the test set
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])

# Print metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

Accuracy: 0.7388
Precision: 0.4324
F1 Score: 0.5089
Recall: 0.6184
AUC: 0.7602


In [25]:
model = RandomForestClassifier()

# Define hyperparameter grid
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}


# Perform grid search with cross-validation
grid_search = RandomizedSearchCV(estimator=model, param_distributions=rf_params, cv=3,random_state=42, n_jobs=-1)
grid_search.fit(X_rus, y_rus)

# Best hyperparameters
best_params = grid_search.best_params_
print(f"Best parameters found: {best_params}")

Best parameters found: {'n_estimators': 1000, 'min_samples_split': 20, 'max_features': 5, 'max_depth': 15}


In [26]:
# Define and fit the Random Forest Classifier
rf2=RandomForestClassifier(n_estimators=1000,min_samples_split=20,max_features=5,max_depth=15)
rf2.fit(x_new,y_new)

y_pred=rf2.predict(x_test) # Make predictions on the test set
model_test_accuracy = accuracy_score(y_test, y_pred) # Calculate Accuracy
model_test_f1 = f1_score(y_test, y_pred, average='weighted') # Calculate F1-score
model_test_precision = precision_score(y_test, y_pred) # Calculate Precision
model_test_recall = recall_score(y_test, y_pred) # Calculate Recall
model_test_rocauc_score = roc_auc_score(y_test, y_pred) #Calculate Roc
print(f"Accuracy:{model_test_accuracy:.4f}")
print(f"F1 score:{model_test_f1:.4f}")
print(f"Precision:{model_test_precision:.4f}")
print(f"Recall:{model_test_recall:.4f}")
print(f"ROC:{model_test_rocauc_score:.4f}")

Accuracy:0.7792
F1 score:0.7836
Precision:0.4959
Recall:0.5560
ROC:0.6988


In [ ]:
#We observe a significant increase in Recall after undersampling